# Impact of socio-economic factors on real estate prices in New York City

## Introduction

Real estate prices in New York City are influenced by a complex interplay of geographic, economic, and social variables. This analysis shows the socio-economic status (SES) of the population and property market values. While location is always important, we can now use data to look deeper into each neighborhood.
By combining real estate transaction data with US Census demographics, this study moves beyond traditional property valuation. We aim to understand why certain areas have higher prices by exploring how variables such as income, professional employment, and commute act as the primary drivers of real estate pricing. We will analyze these trends across all five boroughs: Manhattan, The Bronx, Brooklyn, Queens, and Staten Island.

## Data integration and methodology
To achieve a comprehensive analysis this study merges two datasets:
1. NYC Sales Dataset 2016-2017. It containing transaction prices, property types, and geographical identifiers.
2. US Census Data 2013-2017. It providing demographic and socio-economic metrics at the census tract level.
The connection between these datasets is facilitated through the **Geographic Correspondence** engine allowing us to map census tracts to ZIP codes (ZCTAs) for a precise merge.

**Temporal Alignment (ACS 2017)** \
I use the **ACS 2017 5-year estimates** for the socio-economic part. Even though the property sales data covers a specific period September 2016 - September 2017, the government doesn't publish monthly population data for small neighborhoods. However neighborhood characteristics like income and professional status don't change drastically in a few months. Therefore the 2017 dataset provides a perfect basis for undestanding of the conditions at the time these properties were being sold.

## Analytical framework
The research follows a clear step-by-step process:
1. Data preprocessing \
Cleaning and merging datasets to ensure integrity and alignment. А key step here is Log transformation to handle the right-skewed nature of real estate prices:
$$ \ln(P) = \text{log\_sale\_price} $$

3. Exploratory Data Analysis (EDA) \
Using histograms and scatter plots to visualize trends and relationships. We also use the Pearson correlation coefficient (***r***) to measure the strength and direction of the linear relationship between socio-economic variables and property prices:
$$ r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}} $$

5. Statistical validation \
Applying Levene’s test to check for homogeneity of variance (homoscedasticity):
$$ H_0: \sigma_1^2 = \sigma_2^2 = \dots = \sigma_k^2 $$
We also use Q-Q plots to verify if our log-transformed data follows a normal distribution.

6. Hypothesis testing \
We use Two-Way ANOVA to see if the Borough and Tax group have a real impact on prices. This is measured by the F-statistic, which compares the differences between groups to the variation within them: Utilizing T-tests and two-way ANOVA to determine how different factors affect the price. 
$$ F = \frac{\text{Variance between boroughs}}{\text{Variance within boroughs}} $$
A high F-value will prove that location and building type are significant drivers of the NYC real estate market. We also use independent T-tests to compare specific pairs of boroughs. This helps us see if the price difference between two areas is statistically significant

7. Bayesian inference \
We use Bayes' theorem to calculate probabilities. For example, if we know a property is "Elite" (in the top 10% of the market), we calculate the probability that it is located in a specific borough:
$$ P(\text{Borough} | \text{Elite}) = \frac{P(\text{Elite} | \text{Borough}) \cdot P(\text{Borough})}{P(\text{Elite})} $$

8. Fourier аnalysis \
We use the Fourier transform to see if the housing market follows any waves or cycles (seasonality) over time:
$$ \hat{f}(\xi) = \int_{-\infty}^{\infty} f(x) e^{-2\pi i x \xi} dx $$


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Data cleaning of first dataset - NYC property sales

In [3]:
nyc_dataset = pd.read_csv("nyc-rolling-sales.csv")

In [5]:
nyc_dataset

,Unnamed: 0,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AT PRESENT,BLOCK,LOT,EASE-MENT,BUILDING CLASS AT PRESENT,ADDRESS,...,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE PRICE,SALE DATE
0,4,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,392,6,,C2,153 AVENUE B,...,5,0,5,1633,6440,1900,2,C2,6625000,2017-07-19 00:00:00
1,5,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,399,26,,C7,234 EAST 4TH STREET,...,28,3,31,4616,18690,1900,2,C7,-,2016-12-14 00:00:00
2,6,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,399,39,,C7,197 EAST 3RD STREET,...,16,1,17,2212,7803,1900,2,C7,-,2016-12-09 00:00:00
3,7,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,402,21,,C4,154 EAST 7TH STREET,...,10,0,10,2272,6794,1913,2,C4,3936272,2016-09-23 00:00:00
4,8,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,404,55,,C2,301 EAST 10TH STREET,...,6,0,6,2369,4615,1900,2,C2,8000000,2016-11-17 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84543,8409,5,WOODROW,02 TWO FAMILY DWELLINGS,1,7349,34,,B9,37 QUAIL LANE,...,2,0,2,2400,2575,1998,1,B9,450000,2016-11-28 00:00:00
84544,8410,5,WOODROW,02 TWO FAMILY DWELLINGS,1,7349,78,,B9,32 PHEASANT LANE,...,2,0,2,2498,2377,1998,1,B9,550000,2017-04-21 00:00:00
84545,8411,5,WOODROW,02 TWO FAMILY DWELLINGS,1,7351,60,,B2,49 PITNEY AVENUE,...,2,0,2,4000,1496,1925,1,B2,460000,2017-07-05 00:00:00
84546,8412,5,WOODROW,22 STORE BUILDINGS,4,7100,28,,K6,2730 ARTHUR KILL ROAD,...,0,7,7,208033,64117,2001,4,K6,11693337,2016-12-21 00:00:00


In [6]:
nyc_dataset.head()

,Unnamed: 0,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AT PRESENT,BLOCK,LOT,EASE-MENT,BUILDING CLASS AT PRESENT,ADDRESS,...,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE PRICE,SALE DATE
0,4,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,392,6,,C2,153 AVENUE B,...,5,0,5,1633,6440,1900,2,C2,6625000,2017-07-19 00:00:00
1,5,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,399,26,,C7,234 EAST 4TH STREET,...,28,3,31,4616,18690,1900,2,C7,-,2016-12-14 00:00:00
2,6,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,399,39,,C7,197 EAST 3RD STREET,...,16,1,17,2212,7803,1900,2,C7,-,2016-12-09 00:00:00
3,7,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,402,21,,C4,154 EAST 7TH STREET,...,10,0,10,2272,6794,1913,2,C4,3936272,2016-09-23 00:00:00
4,8,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,404,55,,C2,301 EAST 10TH STREET,...,6,0,6,2369,4615,1900,2,C2,8000000,2016-11-17 00:00:00


In [7]:
nyc_dataset.dtypes

Unnamed: 0                         int64
BOROUGH                            int64
NEIGHBORHOOD                      object
BUILDING CLASS CATEGORY           object
TAX CLASS AT PRESENT              object
BLOCK                              int64
LOT                                int64
EASE-MENT                         object
BUILDING CLASS AT PRESENT         object
ADDRESS                           object
APARTMENT NUMBER                  object
ZIP CODE                           int64
RESIDENTIAL UNITS                  int64
COMMERCIAL UNITS                   int64
TOTAL UNITS                        int64
LAND SQUARE FEET                  object
GROSS SQUARE FEET                 object
YEAR BUILT                         int64
TAX CLASS AT TIME OF SALE          int64
BUILDING CLASS AT TIME OF SALE    object
SALE PRICE                        object
SALE DATE                         object
dtype: object

In [8]:
nyc_dataset.shape

(84548, 22)

In [9]:
nyc_dataset.columns = nyc_dataset.columns.str.lower().str.replace(' ', '_')

In [10]:
nyc_dataset['unnamed:_0']

0           4
1           5
2           6
3           7
4           8
         ... 
84543    8409
84544    8410
84545    8411
84546    8412
84547    8413
Name: unnamed:_0, Length: 84548, dtype: int64

In [11]:
nyc_dataset.drop(columns = ['unnamed:_0'], inplace = True)

In [12]:
nyc_dataset.shape

(84548, 21)

In [15]:
nyc_dataset.isnull().sum()

borough                           0
neighborhood                      0
building_class_category           0
tax_class_at_present              0
block                             0
lot                               0
ease-ment                         0
building_class_at_present         0
address                           0
apartment_number                  0
zip_code                          0
residential_units                 0
commercial_units                  0
total_units                       0
land_square_feet                  0
gross_square_feet                 0
year_built                        0
tax_class_at_time_of_sale         0
building_class_at_time_of_sale    0
sale_price                        0
sale_date                         0
dtype: int64

Some cells maybe contain spaces or dashes ('-') instead of being empty. We need replace them with NaN values to reveal the count of an actual missing data.

In [16]:
nyc_dataset = nyc_dataset.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

In [20]:
pd.set_option('future.no_silent_downcasting', True)

In [21]:
nyc_dataset.replace(['-', '', ' '], np.nan, inplace = True)

I set the ***future.no_silent_downcasting*** option to True. This ensures the code remains compatible with future versions of Pandas and prevents unnecessary warning messages when replacing values.

In [18]:
nyc_dataset = nyc_dataset.infer_objects(copy=False)

In [19]:
print(nyc_dataset.isnull().sum())

borough                               0
neighborhood                          0
building_class_category               0
tax_class_at_present                738
block                                 0
lot                                   0
ease-ment                         84548
building_class_at_present           738
address                               0
apartment_number                  65496
zip_code                              0
residential_units                     0
commercial_units                      0
total_units                           0
land_square_feet                  26252
gross_square_feet                 27612
year_built                            0
tax_class_at_time_of_sale             0
building_class_at_time_of_sale        0
sale_price                        14561
sale_date                             0
dtype: int64


The columns ***land_square_feet***, ***gross_square_feet***, and ***sale_price*** are currently stored as objects. We need to convert them to numeric types to perform mathematical operations, such as calculating averages or analyzing price trends.

In [22]:
num_cols = ['land_square_feet', 'gross_square_feet', 'sale_price']
for col in num_cols:
    nyc_dataset[col] = pd.to_numeric(nyc_dataset[col], errors = 'coerce')

In [23]:
nyc_dataset.dtypes

borough                             int64
neighborhood                       object
building_class_category            object
tax_class_at_present               object
block                               int64
lot                                 int64
ease-ment                         float64
building_class_at_present          object
address                            object
apartment_number                   object
zip_code                            int64
residential_units                   int64
commercial_units                    int64
total_units                         int64
land_square_feet                  float64
gross_square_feet                 float64
year_built                          int64
tax_class_at_time_of_sale           int64
building_class_at_time_of_sale     object
sale_price                        float64
sale_date                          object
dtype: object

In [26]:
nyc_dataset.sale_price

0         6625000.0
1               NaN
2               NaN
3         3936272.0
4         8000000.0
            ...    
84543      450000.0
84544      550000.0
84545      460000.0
84546    11693337.0
84547       69300.0
Name: sale_price, Length: 84548, dtype: float64

Deleting invalid price values -> ***sale_price < 10 000 & sale_price = 0***, because these values are not real sales. They are inherited estates or business sales. They don't have analytical value for our purpose. Also deleting NaN values, as they would distort with our calculations and could lead to biased or incorrect statistical results.

In [27]:
nyc_dataset.dropna(subset = ['sale_price'], inplace = True)

In [28]:
nyc_dataset.sale_price.isnull().sum()

np.int64(0)

In [29]:
(nyc_dataset.sale_price == 0).sum()

np.int64(10228)

In [30]:
(nyc_dataset.sale_price < 1000).sum()

np.int64(11306)

In [31]:
nyc_dataset = nyc_dataset[nyc_dataset.sale_price >= 10000].copy()

In [32]:
nyc_dataset.shape

(58465, 21)

There are residential and commerce buildings. For our analysis we need residential buildings. We need to remove other. For that purpose we can filter the data with ***tax_building_at_sale_price*** to remove business deals.
#### NYC Property Tax Classes
Class 1 - 1-3 Unit residential: 1- to 3-family homes, small condos, and vacant land zoned residential. These are generally assessed at a lower percentage of market value (6%). \
Class 2 - Apartments & co-ops: Rental apartment buildings, cooperatives, and condominiums with 4 or more units. \
Class 3 - Utility properties: Property owned by utility companies, including special franchise property. \
Class 4 - Commercial & industrial: Office buildings, factories, retail space, and warehouses.

In [34]:
nyc_dataset = nyc_dataset[nyc_dataset['tax_class_at_time_of_sale'].isin([1, 2])].copy()

Properties with more than three units are typically commercial investments rather than individual homes. To better analyze factors affecting the average person, we have restricted the dataset to properties with 3 units or fewer. This ensures that institutional investments do not skew our findings

In [36]:
nyc_dataset = nyc_dataset[nyc_dataset.total_units <= 3].copy()

In [37]:
len(nyc_dataset)

53665

In [33]:
pd.qcut(nyc_dataset['sale_price'], 10).value_counts().sort_index()

sale_price
(9999.999, 223294.0]         5847
(223294.0, 335000.0]         5963
(335000.0, 435000.0]         5824
(435000.0, 530000.0]         5949
(530000.0, 640000.0]         5774
(640000.0, 766659.0]         5722
(766659.0, 950000.0]         6066
(950000.0, 1300000.0]        5728
(1300000.0, 2300000.0]       5827
(2300000.0, 2210000000.0]    5765
Name: count, dtype: int64

Refining the dataset to focus on individual residential sales. By selecting Tax Classes 1 and 2 and limiting the properties to 3 units or fewer, we exclude large-scale commercial investments and massive apartment complexes. This leaves us with 53,665 records representing the primary housing market. High-value outliers (up to $98M) are intentionally retained at this stage to observe the full diversity of the NYC luxury sector before final statistical normalization.

The column ease-ment contains no useful information, it is entirely empty, so we are removing it from the dataset.

In [38]:
nyc_dataset = nyc_dataset.drop(columns=['ease-ment']).copy()

The column apartment_number is missing more than 70% of its data. Since it doesn't provide significant value, we have decided to remove it.

In [39]:
nyc_dataset = nyc_dataset.drop(columns = ['apartment_number']).copy()

In [40]:
(nyc_dataset.land_square_feet == 0).sum()

np.int64(7799)

Filling missing values in column ***gross_square_feet*** using the median value for each specific neighborhood. The median is preferred over the mean because real estate data is often skewed by extreme outliers like very large or very small properties.This gives us a more realistic view of a typical property and prevents our analysis from being distorted.

In [41]:
nyc_dataset.land_square_feet = nyc_dataset.groupby('neighborhood')['land_square_feet'].transform(lambda x: x.fillna(x.median()))

In [42]:
nyc_dataset.land_square_feet.isnull().sum()

np.int64(3252)

After the first step 3,252 missing values remain. This happens because some entire neighborhoods have no data at all for square foot, making it impossible to calculate a local median. In this situation, we can use the median of the entire borough instead. This is a broader category that helps us fill the remaining gaps.